# Image FetchPush conedir — OFFLINE ORIGINAL-STYLE qualification (fixed alpha=0 + BC 0.05 + twin critics)

One-seed (seed 0) **strictly offline** qualification of the *original-style* actor on pixels:

```
actor_loss = 0.05 * BC_NLL(data_action | state, future_goal)
           + 0.95 * (-min_Q(state, policy_action, future_goal))
```

- `entropy_coefficient = 0.0`  → **fixed alpha=0** (NO adaptive-alpha optimizer)
- `bc_coef = 0.05`             → 5% goal-conditioned BC anchor, 95% critic actor term
- `twin_q = True`              → actor uses the pessimistic **min(Q1,Q2)**
- `random_goals = 0.0`         → positives from the **same offline trajectory**
- frozen `.npz` IS the buffer  → **zero env interaction** during training (eval-only env)

All eval distances are **physical object–goal coordinates** (sim), never flattened image-L2.

Pipeline: pinned install → clone+checkout → mount Drive → env/render audit →
**reuse the existing frozen dataset** → `scripts/qualify_image_offline_original_style` runs the
preflight + 100k offline run + physical fixed-eval + checkpoint diagnostics + rollout GIFs →
verdict. Writes to a **NEW** run dir; never overwrites prior runs.

> ⚠️ This notebook requires the instrumentation + driver added in this work
> (`scripts/qualify_image_offline_original_style.py`, physical eval + extended logging in
> `crl/train.py` / `crl/losses.py` / `crl/config.py` / `crl/checkpoint.py`). **Commit & push
> those to the fork and set `COMMIT` below to that commit** before running (Cell 2 asserts the
> driver is present).

In [ ]:
# 1. Dependencies WITHOUT disturbing Colab's preinstalled GPU JAX (proven-safe recipe).
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["MUJOCO_GL"] = "egl"                         # headless render backend on Colab
import jax, jaxlib, numpy, subprocess, sys
hold = [f"jax=={jax.__version__}", f"jaxlib=={jaxlib.__version__}", f"numpy=={numpy.__version__}"]
def pip(*a): subprocess.run([sys.executable,"-m","pip","install","-q",*a], check=True)
print("Colab JAX", jax.__version__, "| devices:", jax.devices())
pip("--no-deps","dm-haiku","optax","chex")
pip("jmp","tabulate","toolz","etils","tensorboardX","gymnasium-robotics","mujoco","imageio-ffmpeg",*hold)
print("post-install JAX", jax.__version__, "| devices:", jax.devices())

In [ ]:
# 2. Clone the fork and checkout the commit that CONTAINS the instrumentation + driver.
import os
os.chdir("/content")
if not os.path.exists("/content/contrastive_rl"):
    !git clone https://github.com/tingrui-huang/contrastive_rl.git
%cd /content/contrastive_rl
COMMIT = "REPLACE_WITH_PUSHED_COMMIT"   # <-- set to the commit you pushed with the driver
!git fetch -q origin && git checkout -q $COMMIT
!git log -1 --oneline
assert os.path.exists("scripts/qualify_image_offline_original_style.py"),     "checkout is MISSING the qualification driver -- push it and set COMMIT correctly."

In [ ]:
# 3. Mount Google Drive (checkpoints + artifacts saved here).
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 4. Paths. Reuse the project's EXISTING frozen dataset; write to a NEW run dir.
import os, json, numpy as np
ENV  = "fetch_push_image_conedir"
SEED = 0
SMOKE = True     # True = tiny integration smoke; set False for the real 100k qualification

BASE = "/content/drive/MyDrive/easypush_image_qual"
# The SAME fixed frozen dataset used by the project (do NOT regenerate/alter it).
DATA = f"{BASE}/data/push_image_conedir_noisy_oracle_s{SEED}" + ("_smoke" if SMOKE else "") + ".npz"
# NEW run dir -- must not collide with fetch_push_image_conedir_s0 (online alpha0),
# push_offline_smoke (adaptive), or any state run.
RUN_DIR = f"{BASE}/fetch_push_image_conedir_off_bc005_alpha0_twin_s{SEED}" + ("_smoke" if SMOKE else "")
OUT = "artifacts/image_conedir_offline_original_style"
STEPS = 2000 if SMOKE else 100000
assert not os.path.exists(os.path.join(RUN_DIR, "final.pkl")),     f"{RUN_DIR} already has a finished run; choose a fresh dir rather than overwrite."
print("DATA   :", DATA, "(exists)" if os.path.exists(DATA) else "(MISSING)")
print("RUN_DIR:", RUN_DIR)
assert os.path.exists(DATA), "fixed frozen dataset not found on Drive -- point DATA at it."

In [ ]:
# 5. Environment/render audit -- STOP before training if it fails.
!python -m scripts.smoke_image_conedir
audit = json.load(open("artifacts/push_image_probe/smoke_image_conedir.json"))
assert audit["verdict"] == "PASS", "image-conedir env audit FAILED -- stopping."
print("env audit PASS (oracle=%.2f random=%.2f)" % (
    audit["gate4_oracle"]["oracle_success"], audit["gate4_oracle"]["random_success"]))

In [ ]:
# 6. Run the qualification driver: preflight -> 100k offline -> physical fixed-eval ->
#    checkpoint diagnostics (20k/50k/100k) -> rollout GIFs -> verdict. Physical metrics only.
#    Resume-safe: re-running with the same RUN_DIR continues from latest.pkl (add --resume).
cmd = [
    "python","-m","scripts.qualify_image_offline_original_style",
    "--dataset", DATA, "--run_dir", RUN_DIR, "--out", OUT,
    "--seed", str(SEED), "--steps", str(STEPS),
]
if SMOKE: cmd.append("--smoke")
print(" ".join(cmd))
import subprocess, sys
subprocess.run(cmd, check=True)

In [ ]:
# 7. Show results: preflight, verdict, physical fixed-eval, checkpoint diagnostics, GIFs.
import json, os
from IPython.display import Image, display
print("=== PREFLIGHT ==="); print(json.dumps(json.load(open(f"{OUT}/preflight_audit.json"))["checks"], indent=2))
summ = json.load(open(f"{OUT}/summary.json"))
print("
=== VERDICT:", summ["qualification_verdict"], "===")
print("final physical:", json.dumps(summ["final_checkpoint_physical"], indent=2))
print("
=== fixed_eval.csv ==="); print(open(f"{OUT}/fixed_eval.csv").read())
print("=== checkpoint_diagnostics.csv ==="); print(open(f"{OUT}/checkpoint_diagnostics.csv").read())
for s in ("20000","50000","100000"):
    g = f"{RUN_DIR}/rollout_{s}.gif"
    if os.path.exists(g): display(Image(g, width=256))